# Fase 4 — Figuras e diagnósticos da dissertação

Reorganização do notebook original. Três mudanças em relação à versão do artigo:

1. **Saída em PDF vetorial**, além do PNG. O LaTeX escala PDF sem perda; PNG a 300 dpi
   fica visivelmente pior quando o `\includegraphics` reduz a figura.
2. **Largura fixada na largura do texto da dissertação** (`LARGURA_TEXTO`). As figuras do
   artigo tinham 13–14 polegadas de largura; ao caber numa página A4 de 16 cm elas são
   reduzidas a ~45%, e as fontes de 6–7 pt viram 3 pt, ilegíveis em papel. Os painéis
   passaram de 4×3 para 3×4 (retrato) e as fontes foram redimensionadas.
3. **Célula de diagnóstico** que imprime todos os números pendentes do Capítulo 4.

Ordem de execução: rode a Célula 1 (setup) e a Célula 2 (carga) uma vez; as demais são
independentes entre si.

| Célula | Produz | Vai para |
|---|---|---|
| 3 | Diagnóstico numérico | Capítulo 4 (preenche os `XXXX`) |
| 4 | `cap4_caracterizacao` | Capítulo 4, Figura 2 |
| 5 | `cap5_deslocamento_boxplot` | Capítulo 5 |
| 6 | `cap5_deslocamento_histogramas` | Capítulo 5 |
| 7 | `cap5_mapa_espacial` | Capítulo 5 |
| 8 | `cap6_tradeoff` | Capítulo 6 |
| 9 | `cap6_colapso_dp_a4` | Capítulo 6 |

In [ ]:
# ============================================================
# CÉLULA 1 — SETUP
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

BASE_ANONIMIZADO = "/content/drive/MyDrive/Mestrado/Dados_Anonimizados"
BASE_OUT_FASE2   = "/content/drive/MyDrive/Mestrado/Resultados_Fase2"
BASE_OUT_FASE3   = "/content/drive/MyDrive/Mestrado/Resultados_Fase3"
BASE_FIGURAS     = "/content/drive/MyDrive/Mestrado/Figuras_Dissertacao"
os.makedirs(BASE_FIGURAS, exist_ok=True)

LAT_COL = "latitude"
LON_COL = "longitude"
ANOS    = [2019, 2020, 2021, 2022, 2023]

# Largura útil do texto na dissertação (abnTeX2, A4, margens 3/2 cm) = 16 cm.
# Toda figura é gerada exatamente nessa largura, para entrar no LaTeX com
# \includegraphics[width=\textwidth]{...} SEM reescala. É isso que garante que
# 8 pt na figura seja 8 pt na página.
LARGURA_TEXTO = 16 / 2.54   # ~6.30 polegadas

mpl.rcParams.update({
    'font.family'   : 'serif',
    'font.size'     : 9,
    'axes.labelsize': 9,
    'axes.titlesize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi'    : 110,
    'savefig.dpi'   : 300,
    'savefig.bbox'  : 'tight',
    'axes.grid'     : True,
    'grid.alpha'    : 0.3,
    'pdf.fonttype'  : 42,   # fontes embutidas como TrueType (exigência de algumas bancas)
    'ps.fonttype'   : 42,
})

ORDEM_TECNICAS = [
    "permutacao", "generalizacao_dec2",
    "microagregacao_k2", "microagregacao_k5", "microagregacao_k10",
    "dp_eps_0.1", "dp_eps_0.5", "dp_eps_1.0", "dp_eps_2.0", "dp_eps_5.0",
]

CORES = {
    "original":           "#000000",
    "permutacao":         "#1f77b4",
    "generalizacao_dec2": "#ff7f0e",
    "microagregacao_k2":  "#90c590",
    "microagregacao_k5":  "#4d934d",
    "microagregacao_k10": "#1a521a",
    "dp_eps_0.1":         "#fcd5b0",
    "dp_eps_0.5":         "#fcaa6c",
    "dp_eps_1.0":         "#d62728",
    "dp_eps_2.0":         "#a01a1f",
    "dp_eps_5.0":         "#680f12",
}

NOMES = {
    "original":           "Original",
    "permutacao":         "Permutação",
    "generalizacao_dec2": "Gen. $d$=2",
    "microagregacao_k2":  "Microag. $k$=2",
    "microagregacao_k5":  "Microag. $k$=5",
    "microagregacao_k10": "Microag. $k$=10",
    "dp_eps_0.1":         r"DP $\varepsilon$=0,1",
    "dp_eps_0.5":         r"DP $\varepsilon$=0,5",
    "dp_eps_1.0":         r"DP $\varepsilon$=1,0",
    "dp_eps_2.0":         r"DP $\varepsilon$=2,0",
    "dp_eps_5.0":         r"DP $\varepsilon$=5,0",
}


def haversine_m(lat1, lon1, lat2, lon2):
    R = 6_371_000.0
    lat1, lon1, lat2, lon2 = map(np.deg2rad, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


def salvar(fig, nome):
    """Salva em PDF (para o LaTeX) e PNG (para conferir rápido)."""
    pdf = f"{BASE_FIGURAS}/{nome}.pdf"
    png = f"{BASE_FIGURAS}/{nome}.png"
    fig.savefig(pdf)
    fig.savefig(png)
    print(f"   salvo: {nome}.pdf  +  .png   ({fig.get_size_inches()[0]:.2f}in de largura)")


def carregar_tecnica(tec, seed=0):
    dfs = []
    for ano in ANOS:
        df = pd.read_parquet(f"{BASE_ANONIMIZADO}/{tec}/{ano}.parquet")
        if "seed" in df.columns:
            df = df[df["seed"] == seed]
        dfs.append(df.reset_index(drop=True))
    return pd.concat(dfs, ignore_index=True)


print("Setup pronto.")
print(f"  Largura de figura: {LARGURA_TEXTO:.2f} in ({LARGURA_TEXTO*2.54:.1f} cm)")
print(f"  Saída: {BASE_FIGURAS}")

In [ ]:
# ============================================================
# CÉLULA 2 — CARGA ÚNICA
# Carrega o conjunto original e os deslocamentos de cada técnica.
# Rode uma vez; as células seguintes reaproveitam estas variáveis.
# ============================================================

print("Carregando conjunto original...")
df_orig = carregar_tecnica("original")
print(f"   {len(df_orig):,} registros".replace(",", "."))
print(f"   colunas: {list(df_orig.columns)}")

print("\nCalculando deslocamentos por técnica...")
dados_desloc = {}
for tec in ORDEM_TECNICAS:
    df_anon = carregar_tecnica(tec)
    if len(df_anon) != len(df_orig):
        print(f"   AVISO {tec}: {len(df_anon)} linhas vs {len(df_orig)} no original")
    d = haversine_m(df_orig[LAT_COL].values, df_orig[LON_COL].values,
                    df_anon[LAT_COL].values, df_anon[LON_COL].values)
    dados_desloc[tec] = d[d > 0]
    print(f"   {tec:22s} mediana = {np.median(dados_desloc[tec]):8.0f} m")

print("\nPronto.")

In [ ]:
# ============================================================
# CÉLULA 3 — DIAGNÓSTICO DO CONJUNTO (números do Capítulo 4)
# Imprime tudo o que está marcado como XXXX ou PENDENTE no capítulo.
# Copie a saída direto para o texto.
# ============================================================

print("=" * 66)
print("DIAGNÓSTICO — CAPÍTULO 4")
print("=" * 66)

# ---- detecção de colunas (os nomes variam entre versões do pipeline) ----
def achar(cands, cols):
    for c in cands:
        if c in cols:
            return c
    for c in cols:
        for p in cands:
            if p.lower() in c.lower():
                return c
    return None

cols = list(df_orig.columns)
COL_DATA  = achar(["inspection_realized_at", "realized_at", "data_inspecao"], cols)
COL_QTD   = achar(["total_aedes_aegypti", "aedes_aegypti", "qtd_aedes"], cols)
COL_TRAP  = achar(["trap_id", "armadilha_id", "id_armadilha"], cols)
COL_STAT  = achar(["inspection_status", "status"], cols)

print(f"\ncolunas detectadas:")
print(f"   data     -> {COL_DATA}")
print(f"   qtd      -> {COL_QTD}")
print(f"   trap_id  -> {COL_TRAP}")
print(f"   status   -> {COL_STAT}")

# ---------------------------------------------------------------
# 1. Contagens básicas
# ---------------------------------------------------------------
print("\n" + "-" * 66)
print("1. CONTAGENS")
print("-" * 66)
print(f"   registros geograficamente válidos ......... {len(df_orig):,}".replace(",", "."))
if COL_QTD:
    print(f"   exemplares de Aedes aegypti ............... {int(df_orig[COL_QTD].sum()):,}".replace(",", "."))

pos = df_orig[[LAT_COL, LON_COL]].drop_duplicates()
print(f"   posições geográficas distintas ............ {len(pos):,}".replace(",", "."))
if COL_TRAP:
    print(f"   trap_id distintos ......................... {df_orig[COL_TRAP].nunique():,}".replace(",", "."))
    razao = df_orig[COL_TRAP].nunique() / len(pos)
    print(f"   razão trap_id / posições .................. {razao:.4f}")
    if abs(razao - 1) > 0.01:
        print("   >> ATENÇÃO: os dois números divergem. Isso significa que agrupar por")
        print("      coordenada NÃO equivale a agrupar por armadilha. A Seção 4.3.6 da")
        print("      dissertação precisa registrar essa diferença explicitamente.")
    else:
        print("   >> coordenada e trap_id são equivalentes como chave de agrupamento.")
else:
    print("   trap_id ausente no parquet — a Seção 4.3.6 deve dizer que o agrupamento")
    print("   é necessariamente por coordenada, por indisponibilidade do identificador.")

# ---------------------------------------------------------------
# 2. Precisão espacial da fonte (Seção 4.3.3)
# ---------------------------------------------------------------
print("\n" + "-" * 66)
print("2. PRECISÃO ESPACIAL DA PUBLICAÇÃO")
print("-" * 66)
for nome, v in [("latitude", df_orig[LAT_COL].values), ("longitude", df_orig[LON_COL].values)]:
    ok3 = np.isclose(v, np.round(v, 3), atol=1e-9).mean()
    ok4 = np.isclose(v, np.round(v, 4), atol=1e-9).mean()
    print(f"   {nome:9s}: múltiplos de 0,001 -> {ok3*100:6.2f}%   |  de 0,0001 -> {ok4*100:6.2f}%")
if np.isclose(df_orig[LAT_COL].values, np.round(df_orig[LAT_COL].values, 3), atol=1e-9).mean() > 0.999:
    lat_ref = np.deg2rad(df_orig[LAT_COL].median())
    cel_lat = 0.001 * 111_320
    cel_lon = 0.001 * 111_320 * np.cos(lat_ref)
    print(f"\n   >> CONFIRMADO: 3 casas decimais em toda a série.")
    print(f"      célula = {cel_lat:.1f} m x {cel_lon:.1f} m ; meia-diagonal = "
          f"{np.hypot(cel_lat/2, cel_lon/2):.1f} m")
else:
    print("\n   >> A resolução NÃO é uniformemente de 3 casas. Reveja a Seção 4.3.3.")

# ---------------------------------------------------------------
# 3. Regime de reinspeção (Seção 4.3.6)
# ---------------------------------------------------------------
print("\n" + "-" * 66)
print("3. REGIME DE REINSPEÇÃO")
print("-" * 66)
if COL_DATA:
    d = df_orig.copy()
    dt = pd.to_datetime(d[COL_DATA], format="%d/%m/%Y %H:%M", errors="coerce")
    if dt.isna().mean() > 0.5:
        dt = pd.to_datetime(d[COL_DATA], errors="coerce")
    d["_dt"] = dt
    n_nat = int(d["_dt"].isna().sum())
    print(f"   registros com carimbo inválido ............ {n_nat:,} "
          f"({n_nat/len(d)*100:.2f}%)".replace(",", "."))
    dv = d.dropna(subset=["_dt"])
    print(f"   registros com carimbo válido .............. {len(dv):,}".replace(",", "."))
    print(f"   primeira inspeção ......................... {dv['_dt'].min()}")
    print(f"   última inspeção ........................... {dv['_dt'].max()}")

    chave = COL_TRAP if COL_TRAP else [LAT_COL, LON_COL]
    g = dv.groupby(chave)
    cont = g.size()
    cont2 = cont[cont >= 2]
    print(f"\n   pontos com >= 2 inspeções datadas ......... {len(cont2):,}".replace(",", "."))
    print(f"   n_barra (média de reinspeções) ............ {cont2.mean():.1f}")
    print(f"   mediana ................................... {cont2.median():.0f}")
    print(f"   percentil 95 .............................. {np.percentile(cont2, 95):.0f}")

    intervalos = (dv.sort_values("_dt").groupby(chave)["_dt"]
                    .diff().dt.total_seconds() / 86400).dropna()
    print(f"   intervalo médio entre inspeções ........... {intervalos.mean():.1f} dias")
    print(f"   intervalo mediano ......................... {intervalos.median():.1f} dias")
else:
    print("   coluna de data não encontrada no parquet.")

# ---------------------------------------------------------------
# 4. Composição entomológica e operacional
# ---------------------------------------------------------------
print("\n" + "-" * 66)
print("4. COMPOSIÇÃO")
print("-" * 66)
col_f = achar(["aedes_aegypti_femea", "femeas_aegypti", "aegypti_f"], cols)
col_m = achar(["aedes_aegypti_macho", "machos_aegypti", "aegypti_m"], cols)
if col_f and col_m:
    tf, tm = df_orig[col_f].sum(), df_orig[col_m].sum()
    print(f"   fêmeas de A. aegypti ...................... {int(tf):,}".replace(",", "."))
    print(f"   machos de A. aegypti ...................... {int(tm):,}".replace(",", "."))
    print(f"   proporção de fêmeas ....................... {tf/(tf+tm)*100:.2f}%")
    print("   >> use este número na Seção 4.3.5, em vez do da semana 23/2023.")
else:
    print("   discriminação por sexo não está no parquet consolidado.")
    print("   >> se quiser o número para toda a série, extraia dos JSON brutos;")
    print("      senão, mantenha o exemplo da SE 23/2023 e diga que é ilustrativo.")

if COL_STAT:
    print(f"\n   distribuição de inspection_status:")
    for k, v in df_orig[COL_STAT].value_counts(dropna=False).items():
        print(f"      {str(k):>10s}: {v:,}".replace(",", "."))
else:
    print("\n   inspection_status não está no parquet — a taxa de realização de")
    print("   vistorias precisa vir do notebook de consolidação.")

# ---------------------------------------------------------------
# 5. Série semanal e limiar
# ---------------------------------------------------------------
print("\n" + "-" * 66)
print("5. SÉRIE SEMANAL")
print("-" * 66)
if COL_DATA and COL_QTD:
    serie = (dv.set_index("_dt")[COL_QTD].resample("W-SUN").sum())
    print(f"   semanas na série .......................... {len(serie)}")
    print(f"   total municipal: mín={serie.min():.0f}  mediana={serie.median():.0f}  "
          f"máx={serie.max():.0f}")
    n1 = len(serie) // 6
    print(f"   limiar P80 do 1o fold de treino (n={n1}) ... "
          f"{np.percentile(serie.values[:n1], 80):.0f}")
    print(f"   limiar P80 da série completa .............. "
          f"{np.percentile(serie.values, 80):.0f}")
    globals()["serie_semanal"] = serie

print("\n" + "=" * 66)
print("PENDÊNCIAS QUE ESTE DIAGNÓSTICO NÃO RESOLVE")
print("=" * 66)
print("""
As contagens da cascata de filtros (Tabela 1 do Capítulo 4) precisam sair do
notebook de consolidação, porque o parquet 'original' já é pós-filtro. Rode lá
e anote, nesta ordem:

   (a) linhas na extração bruta, SE 1/2018 a SE 23/2023
   (b) linhas após o recorte 2019-2023
   (c) linhas após remover coordenadas nulas
   (d) linhas após descartar sinais geográficos inválidos
       -> anote também a REGRA exata (ex.: lat > 0 ou lon > 0)
   (e) linhas após o corte de 10 km dos limites municipais  [deve dar 235.707]
""")

In [ ]:
# ============================================================
# CÉLULA 4 — FIGURA DO CAPÍTULO 4
# Caracterização espacial e temporal do conjunto.
# (a) posições DISTINTAS de armadilha  (b) série semanal + limiar P80
#
# Importante: o painel (a) plota posições distintas, não inspeções.
# Plotar as 235.707 inspeções sobrepõe ~95 pontos por armadilha e
# produz uma mancha sólida que não mostra a malha.
# ============================================================

pos = df_orig[[LAT_COL, LON_COL]].drop_duplicates()

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(LARGURA_TEXTO, 2.9),
    gridspec_kw={"width_ratios": [1, 1.45]}
)

# ---- (a) malha de armadilhas ----
ax1.scatter(pos[LON_COL], pos[LAT_COL], s=2.5, c="#222222",
            alpha=0.55, linewidths=0)
ax1.set_xlabel("Longitude")
ax1.set_ylabel("Latitude")
ax1.set_title(f"(a) {len(pos)} posições distintas", fontsize=9)
ax1.set_aspect(1 / np.cos(np.radians(float(df_orig[LAT_COL].median()))))
ax1.tick_params(labelsize=7)
ax1.ticklabel_format(useOffset=False, style="plain")

# ---- (b) série semanal ----
serie = globals().get("serie_semanal")
if serie is None:
    raise RuntimeError("Rode a Célula 3 antes desta — ela constrói `serie_semanal`.")

n1 = len(serie) // 6
limiar = np.percentile(serie.values[:n1], 80)

ax2.plot(serie.index, serie.values, lw=0.9, color="#1f4e79",
         label="$X_t^{\\mathrm{total}}$")
ax2.axhline(limiar, ls="--", lw=1.1, color="#c0392b",
            label=f"$\\tau_{{P80}}^{{\\mathrm{{treino}}}}$ = {limiar:.0f}")
ax2.set_xlabel("Semana")
ax2.set_ylabel("$X_t^{\\mathrm{total}}$ (exemplares)")
ax2.set_title("(b) Total semanal municipal de $\\it{Aedes\\ aegypti}$", fontsize=9)
ax2.tick_params(labelsize=7)
ax2.legend(fontsize=7, loc="upper left", framealpha=0.9, edgecolor="gray")

# Uma marca por ano: com marcas semestrais os rótulos se sobrepõem
# na largura de 16 cm e o eixo fica ilegível.
import matplotlib.dates as mdates
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.xaxis.set_minor_locator(mdates.MonthLocator(bymonth=[7]))

fig.tight_layout()
salvar(fig, "cap4_caracterizacao")
plt.show()

In [ ]:
# ============================================================
# CÉLULA 5 — CAPÍTULO 5: boxplot do deslocamento por técnica
# (era fig1_distribuicao_deslocamento)
# ============================================================

fig, ax = plt.subplots(figsize=(LARGURA_TEXTO, 2.9))

posicoes = list(range(len(ORDEM_TECNICAS)))
bp = ax.boxplot([dados_desloc[t] for t in ORDEM_TECNICAS],
                vert=False, positions=posicoes, widths=0.6,
                showfliers=False, patch_artist=True)
for patch, tec in zip(bp['boxes'], ORDEM_TECNICAS):
    patch.set_facecolor(CORES[tec]); patch.set_alpha(0.75)
for elem in ('medians',):
    for line in bp[elem]:
        line.set_color('#ff7f0e'); line.set_linewidth(1.4)

ax.set_yticks(posicoes)
ax.set_yticklabels([NOMES[t] for t in ORDEM_TECNICAS])
ax.set_xscale('log')
ax.set_xlim(10, 200_000)
ax.set_xticks([10, 100, 1000, 10_000, 100_000])
ax.set_xticklabels(['10 m', '100 m', '1 km', '10 km', '100 km'])
ax.set_xlabel("Deslocamento espacial (escala logarítmica)")

for r in (50, 200, 1000):
    ax.axvline(r, color='gray', linestyle=':', linewidth=0.7, alpha=0.7)

# Linha de referência da resolução da própria fonte (3 casas decimais).
# Isso conecta a figura à Seção 4.3.3: nenhum deslocamento abaixo desse
# valor é distinguível do erro de quantização da publicação.
ax.axvline(74, color='#444444', linestyle='-.', linewidth=0.9, alpha=0.85)

ax.legend(handles=[
    Line2D([0], [0], color='gray', ls=':', lw=1,
           label='Raios operacionais dos ataques (50 m, 200 m, 1 km)'),
    Line2D([0], [0], color='#444444', ls='-.', lw=1,
           label='Meia-diagonal da célula de publicação (74 m)'),
], loc='lower right', fontsize=7, framealpha=0.95,
   edgecolor='gray', fancybox=False)

ax.invert_yaxis()
fig.tight_layout()
salvar(fig, "cap5_deslocamento_boxplot")
plt.show()

In [ ]:
# ============================================================
# CÉLULA 6 — CAPÍTULO 5: histogramas de deslocamento
# (era fig5, em grid 4x3 e 14 in de largura)
#
# Reformatada para 3 colunas x 4 linhas na largura do texto. Numa página
# A4 retrato isso ocupa cerca de meia página e as fontes ficam legíveis;
# o grid 4x3 original teria de ser reduzido a ~45% para caber.
# ============================================================

ordem_plot = ["original"] + ORDEM_TECNICAS   # 11 painéis

fig, axes = plt.subplots(4, 3, figsize=(LARGURA_TEXTO, LARGURA_TEXTO * 1.15))
axes = axes.flatten()

for ax, tec in zip(axes, ordem_plot):
    if tec == "original":
        ax.text(0.5, 0.5, "Original\n(referência)\n\nDeslocamento = 0",
                transform=ax.transAxes, ha='center', va='center',
                fontsize=8, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.7))
        ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
        ax.set_title(NOMES[tec], fontsize=8.5, fontweight='bold')
        continue

    dist = dados_desloc[tec]
    ax.hist(dist, bins=40, color=CORES[tec], alpha=0.8,
            edgecolor='white', linewidth=0.25)
    ax.text(0.97, 0.94,
            f"mín {dist.min():.0f} m\nmed {np.median(dist):.0f} m\nmáx {dist.max():.0f} m",
            transform=ax.transAxes, ha='right', va='top', fontsize=6.2,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85,
                      edgecolor='gray', linewidth=0.35))
    ax.set_title(NOMES[tec], fontsize=8.5, fontweight='bold')
    ax.tick_params(labelsize=6)
    ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(
        lambda v, p: f"{v/1000:.0f}k" if v >= 1000 else f"{v:.0f}"))
    ax.grid(True, alpha=0.3)

axes[-1].axis('off')

# Rótulos de eixo apenas na borda, para não poluir 11 painéis
for i, ax in enumerate(axes[:-1]):
    if i % 3 == 0:
        ax.set_ylabel("Contagem", fontsize=7)
    if i >= 8:
        ax.set_xlabel("Distância (m)", fontsize=7)

fig.tight_layout()
salvar(fig, "cap5_deslocamento_histogramas")
plt.show()

In [ ]:
# ============================================================
# CÉLULA 7 — CAPÍTULO 5: distribuição espacial por técnica
# (era fig6_mapa_calor_espacial, em grid 4x3 e 13 in de largura)
#
# Reformatada para 3x4 retrato na largura do texto. Ocupa uma página
# inteira; use \begin{figure}[p] no LaTeX.
# ============================================================

SAMPLE_N = 8000
rng = np.random.default_rng(42)
idx = rng.choice(len(df_orig), size=min(SAMPLE_N, len(df_orig)), replace=False)

lat_o = df_orig[LAT_COL].values[idx]
lon_o = df_orig[LON_COL].values[idx]

print("Amostrando pontos por técnica...")
pontos = {"original": (lat_o, lon_o, np.zeros(len(idx)))}
for tec in ORDEM_TECNICAS:
    df_a = carregar_tecnica(tec)
    la, lo = df_a[LAT_COL].values[idx], df_a[LON_COL].values[idx]
    pontos[tec] = (la, lo, haversine_m(lat_o, lon_o, la, lo))
    print(f"   {tec}")

cmap = plt.cm.RdYlGn_r
norm = Normalize(vmin=0, vmax=4)          # log10 de 1 m a 10 km

# Mesma janela geográfica em todos os painéis: sem isso, cada painel
# reescala e a comparação visual entre técnicas fica enganosa.
m = 0.02
lat_lim = (np.percentile(lat_o, 0.5) - m, np.percentile(lat_o, 99.5) + m)
lon_lim = (np.percentile(lon_o, 0.5) - m, np.percentile(lon_o, 99.5) + m)

ordem_plot = ["original"] + ORDEM_TECNICAS

fig, axes = plt.subplots(4, 3, figsize=(LARGURA_TEXTO, LARGURA_TEXTO * 1.25))
axes = axes.flatten()

for i, (ax, tec) in enumerate(zip(axes, ordem_plot)):
    lat, lon, dist = pontos[tec]
    if tec == "original":
        ax.scatter(lon, lat, s=1.5, alpha=0.5, c='black', edgecolors='none')
    else:
        ax.scatter(lon, lat, s=1.5, alpha=0.6,
                   c=np.log10(np.maximum(dist, 1)),
                   cmap=cmap, norm=norm, edgecolors='none')
        ax.text(0.5, 1.005, f"máx {dist.max():.0f} m", transform=ax.transAxes,
                ha='center', va='bottom', fontsize=6)
    ax.set_title(NOMES[tec], fontsize=8.5, fontweight='bold', pad=11)
    ax.set_xlim(*lon_lim); ax.set_ylim(*lat_lim)
    ax.set_aspect(1 / np.cos(np.radians(float(np.median(lat_o)))))
    ax.tick_params(labelsize=5.5)
    ax.ticklabel_format(useOffset=False, style="plain")
    # Poucas marcas: com o padrão do matplotlib os rótulos de longitude
    # se encavalam na largura de cada painel.
    ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(3))
    ax.yaxis.set_major_locator(mpl.ticker.MaxNLocator(4))
    ax.grid(True, alpha=0.2)
    if i % 3 != 0:
        ax.set_yticklabels([])
    else:
        ax.set_ylabel("Latitude", fontsize=7)
    if i < 8:
        ax.set_xticklabels([])
    else:
        ax.set_xlabel("Longitude", fontsize=7)

axes[-1].axis('off')

sm = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
cbar = fig.colorbar(sm, ax=axes[-1], fraction=0.6, aspect=12, extend='max')
cbar.set_label("Deslocamento (escala log)", fontsize=7)
cbar.set_ticks([0, 1, 2, 3, 4])
cbar.set_ticklabels(['1 m', '10 m', '100 m', '1 km', '10 km'])
cbar.ax.tick_params(labelsize=6)

fig.tight_layout()
salvar(fig, "cap5_mapa_espacial")
plt.show()

In [ ]:
# ============================================================
# CÉLULA 8 — CAPÍTULO 6: plano privacidade x utilidade
# (era fig2_tradeoff_priv_util)
# ============================================================

df_util    = pd.read_parquet(f"{BASE_OUT_FASE2}/ic_por_tecnica.parquet")
df_ataques = pd.read_parquet(f"{BASE_OUT_FASE3}/resultados_ataques.parquet")

util = df_util[df_util['metrica'] == 'auc'].set_index('tecnica')['mediana']
a1 = df_ataques[(df_ataques['ataque'] == 'A1') &
                (df_ataques['nivel_aux'] == 0.10) &
                (df_ataques['valor_param'] == 200)]
priv = 1 - a1.groupby('tecnica')['valor'].median()
a4 = df_ataques[(df_ataques['ataque'] == 'A4') &
                (df_ataques['metrica'] == 'erro_mediano_m')]
err_a4 = a4.groupby('tecnica')['valor'].median()

df_plot = pd.DataFrame({'util': util, 'priv': priv, 'err_a4': err_a4}).dropna()

fig, ax = plt.subplots(figsize=(LARGURA_TEXTO, 3.2))

sc = ax.scatter(df_plot['priv'], df_plot['util'],
                c=np.log10(df_plot['err_a4']), cmap='RdYlGn',
                s=90, edgecolors='black', linewidths=0.5,
                vmin=1.5, vmax=4, zorder=3)

deslocs = {
    'microagregacao_k10': ('right', -0.0006,  0.0025),
    'microagregacao_k5':  ('left',   0.0004, -0.0018),
    'microagregacao_k2':  ('left',   0.0004, -0.0040),
    'permutacao':         ('left',   0.0004,  0.0030),
    'generalizacao_dec2': ('left',   0.0006,  0.0015),
    'dp_eps_0.1':         ('right', -0.0006, -0.0040),
    'dp_eps_0.5':         ('right', -0.0006,  0.0035),
    'dp_eps_1.0':         ('left',   0.0004, -0.0045),
    'dp_eps_2.0':         ('left',   0.0004,  0.0025),
    'dp_eps_5.0':         ('left',   0.0006,  0.0000),
}
for tec, row in df_plot.iterrows():
    ha, dx, dy = deslocs.get(tec, ('left', 0.0004, 0.0025))
    ax.annotate(NOMES[tec], xy=(row['priv'], row['util']),
                xytext=(row['priv'] + dx, row['util'] + dy),
                fontsize=7, ha=ha)

if 'microagregacao_k10' in df_plot.index:
    v = df_plot.loc['microagregacao_k10']
    ax.scatter([v['priv']], [v['util']], s=210, facecolors='none',
               edgecolors='darkgreen', linewidths=1.8, zorder=5)

cbar = plt.colorbar(sc, ax=ax, label='Erro A4 (m, escala log)', shrink=0.9)
cbar.set_ticks([1.7, 2, 2.5, 3, 3.7])
cbar.set_ticklabels(['50 m', '100 m', '316 m', '1 km', '5 km'])

ax.set_xlabel(r"Privacidade $1 - \tau_{A_1}(200\,\mathrm{m})$, vazamento auxiliar de 10\%")
ax.set_ylabel("Utilidade (AUC mediana)")
ax.set_xlim(0.975, 1.005)
ax.set_ylim(0.860, 0.912)

fig.tight_layout()
salvar(fig, "cap6_tradeoff")
plt.show()

In [ ]:
# ============================================================
# CÉLULA 9 — CAPÍTULO 6: colapso da proteção temporal da DP
# (era fig3_dp_colapso_a4)
# ============================================================

df_ataques = pd.read_parquet(f"{BASE_OUT_FASE3}/resultados_ataques.parquet")
a4 = df_ataques[(df_ataques['ataque'] == 'A4') &
                (df_ataques['metrica'] == 'erro_mediano_m')]

eps_dp = [0.1, 0.5, 1.0, 2.0, 5.0]
erros, p95 = [], []
for e in eps_dp:
    tec = f"dp_eps_{e}"
    erros.append(a4[a4['tecnica'] == tec]['valor'].median())
    p95.append(df_ataques[(df_ataques['ataque'] == 'A4') &
                          (df_ataques['metrica'] == 'erro_p95_m') &
                          (df_ataques['tecnica'] == tec)]['valor'].median())

ref = {t: a4[a4['tecnica'] == t]['valor'].median()
       for t in ('microagregacao_k10', 'permutacao', 'generalizacao_dec2')}

fig, ax = plt.subplots(figsize=(LARGURA_TEXTO, 3.0))

ax.plot(eps_dp, erros, 'o-', color='#d62728', lw=1.7, ms=6.5,
        label='DP geográfica (mediana)', zorder=3)
ax.fill_between(eps_dp, erros, p95, color='#d62728', alpha=0.15,
                label='DP, faixa P50--P95')

for tec, cor in [('microagregacao_k10', '#1a521a'),
                 ('permutacao', '#1f77b4'),
                 ('generalizacao_dec2', '#ff7f0e')]:
    ax.axhline(ref[tec], color=cor, ls='--', lw=1.1, alpha=0.85,
               label=f"{NOMES[tec]} ({ref[tec]:.0f} m)")

ax.axhspan(0, 200, color='red', alpha=0.08, label='Zona crítica ($<$200 m)')

for e, v in zip(eps_dp, erros):
    ax.annotate(f'{v:.0f} m', xy=(e, v), xytext=(6, 2),
                textcoords='offset points', fontsize=7,
                color='#d62728', fontweight='bold')

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xticks(eps_dp)
ax.set_xticklabels([str(e).replace('.', ',') for e in eps_dp])
ax.set_xlabel(r"$\varepsilon$ da privacidade diferencial geográfica (km$^{-1}$)")
ax.set_ylabel("Erro mediano do ataque A4 (m, escala log)")
ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=7,
          framealpha=0.95, edgecolor='gray', fancybox=False)
ax.grid(True, which='both', alpha=0.3)

fig.tight_layout()
salvar(fig, "cap6_colapso_dp_a4")
plt.show()

In [ ]:
# ============================================================
# CÉLULA 10 — RESUMO E TRECHOS LaTeX PRONTOS
# ============================================================

import glob
print("Arquivos gerados em", BASE_FIGURAS, "\n")
for f in sorted(glob.glob(f"{BASE_FIGURAS}/*.pdf")):
    print("  ", os.path.basename(f))

print("""

Trechos LaTeX (as figuras já saem na largura exata do texto, então use
sempre width=\\textwidth e nunca um fator de escala):

  % Capítulo 4
  \\includegraphics[width=\\textwidth]{figuras/cap4_caracterizacao}

  % Capítulo 5
  \\includegraphics[width=\\textwidth]{figuras/cap5_deslocamento_boxplot}
  \\includegraphics[width=\\textwidth]{figuras/cap5_deslocamento_histogramas}
  \\includegraphics[width=\\textwidth]{figuras/cap5_mapa_espacial}   % use [p]

  % Capítulo 6
  \\includegraphics[width=\\textwidth]{figuras/cap6_tradeoff}
  \\includegraphics[width=\\textwidth]{figuras/cap6_colapso_dp_a4}
""")